# E08 · 01 单篇 PDF → 完整 MinerU 产出（md + 图 + block + 溯源）

输入：`data/raw/e08/pdf/*.pdf`。输出：`_scratch/ocr/<主名>/` —— 探索产物，
**不是** `data/processed/` 的流水线工件。

    <stem>.md      正文；image/chart 处插 ![](images/...)，table 的裁图用注释指路
    blocks.json    逐页 block 原样（type / 归一化 bbox / angle / content / sub_type）+ 裁图清单
    images/        image / chart / table 块的裁图
    pages/         整页渲染，默认不留（keep_page_renders=True 才有）
    meta.json      溯源：pdf sha256、页数、dpi、服务地址、模型、库版本、耗时、块计数

**口径声明**

- OCR 结果是**机器抽取的文本，不是事实**：表格、公式、图注的还原都可能出错，
  任何要进入论断的数字都应回 PDF 原文核对。同一篇重跑两次不保证逐字相同。
- 图的**描述**来自服务端的 `image_analysis`（默认已开）：flowchart → mermaid、
  chart → markdown 表、pure_table → HTML、natural_image → caption。图的**像素**
  是我们自己按 block 的归一化 bbox 从页图裁的 —— `mineru_vl_utils` 不写任何文件。
  两者都可能错位，`blocks.json` 里留了 bbox 和页号供核对。
- `bbox` 归一化、与 dpi 无关，所以 `crop_dpi` 可以高于 OCR 的 `dpi`，裁图更清晰。
- 服务地址取自 `experiments/e08/.env` 的 `MINERU_SERVICE=`（或同名环境变量，或
  `server_url=` 参数；优先级：参数 > 环境变量 > `.env`）。找不到就抛 `RuntimeError`，
  不会悄悄回落到 localhost —— 所以产物属于哪个服务是确定的。
- 渲染依赖系统二进制 `mutool`（Debian/Ubuntu：`apt install mupdf-tools`）。
- 本 notebook 是**探索面**。若这批产出成为文档引用数字的来源，按 AGENTS.md
  先提升为 `src/e08/pN_*.py`，产物写进 `data/processed/e08/`。

In [ ]:
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "AGENTS.md").exists():          # notebook 可能从任意目录启动
    REPO = REPO.parent

from e08.mineru_ocr import aocr_bundle

PDF_DIR = REPO / "data/raw/e08/pdf"
OUT_ROOT = Path.cwd() / "_scratch/ocr"            # 探索产物，不进 data/processed

pdfs = sorted(PDF_DIR.glob("*.pdf"))
print(f"{len(pdfs)} 篇：")
for p in pdfs[:5]:
    print(" ", p.stem[:70])
print("  ...")

pdf = pdfs[0]                                     # 换成你要看的那篇
assert pdf.is_file(), pdf
pdf.stem

In [ ]:
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "AGENTS.md").exists():          # notebook 可能从任意目录启动
    REPO = REPO.parent

from e08.mineru_ocr import aocr_bundle

PDF_DIR = REPO / "data/raw/e08/pdf"
OUT_ROOT = Path.cwd() / "_scratch/ocr"            # 探索产物，不进 data/processed

pdfs = sorted(PDF_DIR.glob("*.pdf"))
print(f"{len(pdfs)} 篇：")
for p in pdfs[:5]:
    print(" ", p.stem[:70])
print("  ...")

pdf = pdfs[0]                                     # 换成你要看的那篇
assert pdf.is_file(), pdf
pdf.stem

## 跑一篇

单独一个 cell：一篇 20 页上下的论文要几十秒到几分钟。同步版 `ocr_bundle` 在 Jupyter 里
会直接抛 `RuntimeError`（已有事件循环），所以用 `await`。

目录已存在且非空会抛 `FileExistsError` —— 重跑同一篇要 `overwrite=True`，免得手滑重烧一遍。

In [ ]:
bundle = await aocr_bundle(pdf, OUT_ROOT, crop_dpi=300)
bundle.relative_to(REPO)

## 看产出

先看溯源和块计数，再看图,最后看 md 开头。这三样合起来能回答"这篇抽到了什么、图留下了几张"。

In [ ]:
import json

meta = json.loads((bundle / "meta.json").read_text(encoding="utf-8"))
print(f"{meta['page_count']} 页 | {meta['counts']['blocks']} 块 | 裁图 {meta['counts']['crops']} 张 "
      f"| {meta['run']['elapsed_s']}s | dpi {meta['params']['dpi']}/{meta['params']['crop_dpi']}")
print("服务:", meta["service"]["server_url"], "来源:", meta["service"]["url_origin"])
print("块类型:", meta["counts"]["by_type"])

In [ ]:
from IPython.display import Image as IPyImage, display

crops = sorted((bundle / "images").glob("*.png"))
print(f"{len(crops)} 张裁图：", [p.name for p in crops][:10])
for p in crops[:3]:                               # 只看前三张，别把 notebook 撑爆
    print(p.name)
    display(IPyImage(filename=str(p), width=520))

In [ ]:
md_text = (bundle / f"{pdf.stem}.md").read_text(encoding="utf-8")
print(f"{len(md_text)} chars，图片引用 {md_text.count('![')} 处，表格裁图注释 {md_text.count('<!-- crop:')} 处")
print(md_text[:1200])